# 07 · 全量档：ResNet-18 / 完整 CIFAR-10 / 200 epochs

本 Notebook 读取 `reports-full/full_metrics.json`，把 **论文 CIFAR-10 数值**、
**全量档（ResNet-18）实测** 与 **缩比档（SmallCNN）实测** 放在同一张表里。

它回答的是：

> 换成论文的骨干和训练规模之后，EBBA / EBBA+ 的机制指标是否朝论文数值移动？

它**不是**论文表格的逐位复现。攻击实现、OOD 图像、聚类定向规则和随机种子仍与原文不同；
所有差异在最后一节列出。阅读时请先看方向和量级，再看小数点。


In [1]:
from pathlib import Path
import json
import os

from IPython.display import HTML, display

ROOT = Path(os.environ.get("EBBA_ROOT", Path.cwd()))
reports = Path(os.environ.get("EBBA_REPORTS", ROOT / "reports-full"))
metrics_path = reports / "full_metrics.json"

if not metrics_path.exists():
    raise FileNotFoundError(
        f"{metrics_path} 不存在。请先运行全量训练与评估：\n"
        "  docker compose --profile train run --rm ebba-train train-full --all --resume auto\n"
        "  docker compose --profile full run --rm ebba-full verify-full --device cuda"
    )

payload = json.loads(metrics_path.read_text(encoding="utf-8"))
print("profile   :", payload.get("profile"))
print("generated :", payload.get("generated_at"))
print("device    :", payload.get("environment", {}).get("device"))
print("gpu       :", payload.get("environment", {}).get("gpu"))
print("elapsed s :", payload.get("elapsed_seconds"))
print("attacks   :", ", ".join(sorted(payload.get("models", {}))))


profile   : full
generated : 2026-10-21T18:14:16+00:00
device    : cuda
gpu       : NVIDIA GeForce RTX 5060 Laptop GPU
elapsed s : 441.1
attacks   : badnets, blend, clean


## 1. 主结果对照

In [2]:
def fmt(x, digits=3):
    if x is None:
        return "—"
    if isinstance(x, bool):
        return "是" if x else "否"
    if isinstance(x, (int, float)):
        return f"{x:.{digits}f}"
    return str(x)


paper = payload.get("paper", {})
proxy = payload.get("proxy", {})
models = payload.get("models", {})
clustering = payload.get("clustering", {})
removal = payload.get("removal", {})

rows = []
for attack in ("badnets", "blend"):
    p = paper.get(attack, {})
    m = models.get(attack, {})
    px = proxy.get(attack, {})
    c = clustering.get(attack, {})
    primary = c.get("birch_embed", {})
    refined = c.get("refined", {})
    r = removal.get(attack, {})
    rows.append(
        "<tr>"
        f"<td>{attack}</td>"
        f"<td>{fmt(p.get('no_defense_ba'))} / {fmt(p.get('no_defense_asr'))}</td>"
        f"<td>{fmt(m.get('ba'))} / {fmt(m.get('asr'))}</td>"
        f"<td>{fmt(px.get('ba'))} / {fmt(px.get('asr'))}</td>"
        f"<td>{fmt(p.get('pdr'))} / {fmt(p.get('f1'))}</td>"
        f"<td>{fmt(primary.get('pdr'))} / {fmt(primary.get('f1'))}"
        f" <small>(细化 {fmt(refined.get('pdr'))} / {fmt(refined.get('f1'))})</small></td>"
        f"<td>{fmt(px.get('pdr'))} / {fmt(px.get('f1'))}</td>"
        f"<td>{fmt(p.get('ebba_plus_asr'))}</td>"
        f"<td>{fmt(r.get('asr_after_pool'))}"
        f" <small>(细化 {fmt(r.get('asr_after_pool_refined'))})</small></td>"
        f"<td>{fmt(px.get('removal_asr'))}</td>"
        "</tr>"
    )

html = """
<style>
table{border-collapse:collapse;margin:8px 0;font-size:13px}
th,td{border:1px solid #d0d7de;padding:6px 8px;text-align:left}
th{background:#f6f8fa}
small{color:#57606a}
</style>
<table>
<tr><th rowspan="2">攻击</th><th colspan="3">BA / ASR（未防御）</th>
<th colspan="3">PDR / F1</th><th colspan="3">去除后 ASR</th></tr>
<tr><th>论文</th><th>ResNet-18 full</th><th>SmallCNN proxy</th>
<th>论文 EBBA+</th><th>ResNet-18 full</th><th>SmallCNN proxy</th>
<th>论文 EBBA+</th><th>full（池内）</th><th>proxy（全测试估计）</th></tr>
""" + "".join(rows) + "</table>"
display(HTML(html))
print("括号内的“细化”是 P0 加的池内二次阈值（诊断规则，不是论文方法）；")
print("与论文比较时请看括号外的值。")


括号内的“细化”是 P0 加的池内二次阈值（诊断规则，不是论文方法）；
与论文比较时请看括号外的值。


## 怎么读这张表

- **BA / ASR**：实验台是否达到论文量级。论文 No Defense 的 CIFAR-10 是 BA≈91.2–91.4、ASR≈99.4–99.97。
- **PDR / F1**：EBBA+ 在候选池中找出“被攻击的图”的能力，论文是 0.95–0.96。
- **去除后 ASR**：论文用完整测试集；full 档这里记录的是**候选池内**的中毒图残存成功率，
  口径更窄，不能与论文列直接相减。全测试集估计另见 JSON 的 `asr_after_test_estimate`。

下一节把检测、Fig.1、Fig.2 和聚类细节展开。


## 2. 检测与 Fig.1

In [3]:
det = payload.get("detection", {})
det_rows = []
for name, rec in det.items():
    d22 = rec.get("lambda_2.2", {})
    d30 = rec.get("lambda_3.0", {})
    det_rows.append(
        "<tr>"
        f"<td>{name}</td>"
        f"<td>{fmt(d22.get('target'))}</td>"
        f"<td>{fmt(d22.get('zscore'))}</td>"
        f"<td>{fmt(d22.get('is_backdoored'))}</td>"
        f"<td>{fmt(d30.get('zscore'))}</td>"
        f"<td>{fmt(d30.get('is_backdoored'))}</td>"
        "</tr>"
    )
display(HTML(
    "<h3>EBBA 检测（任务无关图像）</h3>"
    "<table><tr><th>模型</th><th>λ=2.2 目标</th><th>λ=2.2 z</th><th>λ=2.2 检出</th>"
    "<th>λ=3 z</th><th>λ=3 检出</th></tr>" + "".join(det_rows) + "</table>"
))

fig1 = payload.get("fig1", {})
rows = []
for name, rec in fig1.items():
    rows.append(
        "<tr>"
        f"<td>{name}</td><td>{fmt(rec.get('top1'))}</td><td>{fmt(rec.get('top3'))}</td>"
        f"<td>{fmt(rec.get('mean_rank'))}</td><td>{fmt(rec.get('uniform_mean_rank'))}</td>"
        "</tr>"
    )
display(HTML(
    "<h3>Fig.1：目标标签排名</h3>"
    "<table><tr><th>模型</th><th>top-1</th><th>top-3</th><th>平均排名</th>"
    "<th>均匀基线</th></tr>" + "".join(rows) + "</table>"
))


模型,λ=2.2 目标,λ=2.2 z,λ=2.2 检出,λ=3 z,λ=3 检出
clean,3.000,2.422,是,2.422,否
badnets,0.000,2.725,是,2.725,否
blend,3.000,2.225,是,2.225,否


模型,top-1,top-3,平均排名,均匀基线
clean,0.086,0.336,4.180,4.500
badnets,0.218,0.678,2.267,4.500
blend,0.108,0.367,3.842,4.500


## 3. Fig.2 与 NTE

In [4]:
fig2 = payload.get("fig2", {})
rows = []
for attack, rec in fig2.items():
    ps = rec.get("probability_shift", {})
    rs = rec.get("rank_shift", {})
    rows.append(
        "<tr>"
        f"<td>{attack}</td>"
        f"<td>{fmt(ps.get('target_decrease_fraction'))}</td>"
        f"<td>{fmt(ps.get('gt_increase_fraction'))}</td>"
        f"<td>{fmt(ps.get('both_fraction'))}</td>"
        f"<td>{fmt(rs.get('gt_forward_fraction'))}</td>"
        f"<td>{fmt(rs.get('target_backward_fraction'))}</td>"
        "</tr>"
    )
display(HTML(
    "<h3>Fig.2：损坏后的概率与排名迁移</h3>"
    "<table><tr><th>攻击</th><th>目标概率下降</th><th>真值概率上升</th><th>两者同时</th>"
    "<th>真值排名前移</th><th>目标排名后移</th></tr>" + "".join(rows) + "</table>"
))


攻击,目标概率下降,真值概率上升,两者同时,真值排名前移,目标排名后移
badnets,0.808,0.677,0.668,0.223,0.358
blend,0.885,0.829,0.801,0.299,0.415


In [5]:
nte = payload.get("nte", {})
rows = []
for attack, rec in nte.items():
    rows.append(
        "<tr>"
        f"<td>{attack}</td><td>{fmt(rec.get('coverage'))}</td>"
        f"<td>{fmt(rec.get('recovery_poisoned'))}</td>"
        f"<td>{fmt(rec.get('recovery_clean_target'))}</td>"
        f"<td>{fmt(rec.get('n_kept'))}</td>"
        "</tr>"
    )
display(HTML(
    "<h3>NTE 标签恢复（Eq.12）</h3>"
    "<table><tr><th>攻击</th><th>候选池覆盖率</th><th>中毒图恢复率</th>"
    "<th>干净目标类恢复率</th><th>进入 EBBA+ 的图数</th></tr>" + "".join(rows) + "</table>"
))

rows = []
for attack, rec in clustering.items():
    primary = rec.get("birch_embed_detail", {})
    refined = rec.get("refined", {})
    stats = refined.get("stats", {}) or {}
    legacy = rec.get("birch_margin_detail", {})
    methods = rec.get("methods", {})
    rows.append(
        "<tr>"
        f"<td>{attack}</td>"
        f"<td>{fmt(primary.get('pdr'))}</td><td>{fmt(primary.get('f1'))}</td>"
        f"<td>{fmt(primary.get('precision'))}</td>"
        f"<td>{fmt(refined.get('pdr'))}</td><td>{fmt(refined.get('f1'))}</td>"
        f"<td>{fmt(refined.get('precision'))}</td><td>{fmt(refined.get('fp'))}</td>"
        f"<td>{fmt(stats.get('applied'))} (ΔBIC {fmt(stats.get('delta_bic'), 1)})</td>"
        f"<td>{fmt(legacy.get('pdr'))}</td>"
        f"<td>{fmt(rec.get('oracle', {}).get('pdr'))}</td>"
        f"<td>{fmt(rec.get('pdr_spread'))}</td>"
        f"<td>{fmt(methods.get('hc', {}).get('pdr'))} / "
        f"{fmt(methods.get('birch', {}).get('pdr'))} / "
        f"{fmt(methods.get('meanshift', {}).get('pdr'))} / "
        f"{fmt(methods.get('dbscan', {}).get('pdr'))}</td>"
        "</tr>"
    )
display(HTML(
    "<h3>EBBA+ 聚类与定向规则</h3>"
    "<table><tr><th rowspan='2'>攻击</th><th colspan='3'>主配置（Birch + 峰值 NTE）</th>"
    "<th colspan='5'>池内二次阈值（诊断，非论文方法）</th><th colspan='4'>对照</th></tr>"
    "<tr><th>PDR</th><th>F1</th><th>精确率</th>"
    "<th>PDR</th><th>F1</th><th>精确率</th><th>FP</th><th>启用 (ΔBIC)</th>"
    "<th>NTE 特征空间 PDR</th><th>Oracle 上界</th><th>四方法极差</th>"
    "<th>HC / Birch / MeanShift / DBSCAN</th></tr>"
    + "".join(rows) + "</table>"
))

# P0：NTE 上限/温度/类别归一化扫描（恢复率用 Eq.12，PDR 用同一 Birch 划分）
variant_names = []
for rec in clustering.values():
    for name in (rec.get("nte_variants") or {}):
        if name not in variant_names:
            variant_names.append(name)
rows = []
for attack, rec in clustering.items():
    variants = rec.get("nte_variants") or {}
    for name in variant_names:
        v = variants.get(name, {})
        rows.append(
            "<tr>"
            f"<td>{attack}</td><td>{name}</td>"
            f"<td>{fmt(v.get('recovery_poisoned'))}</td>"
            f"<td>{fmt(v.get('recovery_clean_target'))}</td>"
            f"<td>{fmt(v.get('pdr'))}</td><td>{fmt(v.get('f1'))}</td>"
            f"<td>{fmt(v.get('precision'))}</td>"
            f"<td>{fmt(v.get('asr_after_pool'))}</td>"
            f"<td>{fmt(v.get('ba_after_pool'))}</td>"
            f"<td>{fmt(v.get('poisoned_cluster'))}</td>"
            "</tr>"
        )
display(HTML(
    "<h3>NTE 上限 / 温度 / 类别归一化扫描</h3>"
    "<table><tr><th>攻击</th><th>变体</th><th>中毒图恢复率</th><th>干净目标类恢复率</th>"
    "<th>PDR</th><th>F1</th><th>精确率</th><th>去除后池内 ASR</th><th>去除后池内 BA</th>"
    "<th>选定簇</th></tr>"
    + "".join(rows) + "</table>"
))

for attack, rec in clustering.items():
    print(attack, "方法实际使用：", {k: v.get("method_used") for k, v in rec.get("methods", {}).items()})
print("主配置：损坏后图像嵌入 + 峰值 NTE 定向；第二列是 NTE 向量特征空间，仅供对照。")
print("cap5_T1 是论文实现（官方 Notebook 的上限 5），uncapped_T1 是 Eq.9-11 的字面形式。")
print("同一簇划分下，去除步骤使用 Eq.12 的恢复标签，所以 ASR/BA 会随归一化变化。")


攻击,候选池覆盖率,中毒图恢复率,干净目标类恢复率,进入 EBBA+ 的图数
badnets,0.979,0.780,0.002,5872.000
blend,0.992,0.362,0.000,5950.000


攻击,变体,中毒图恢复率,干净目标类恢复率,PDR,F1,精确率,去除后池内 ASR,去除后池内 BA,选定簇
badnets,cap5_T1,0.780,0.002,0.891,0.935,0.984,0.109,0.929,0.000
badnets,uncapped_T1,0.780,0.002,0.891,0.935,0.984,0.109,0.929,0.000
badnets,cap5_T2,0.782,0.003,0.891,0.935,0.984,0.109,0.929,0.000
badnets,uncapped_T2,0.785,0.002,0.891,0.935,0.984,0.109,0.929,0.000
badnets,cap5_T1_classcenter,0.833,0.584,0.891,0.935,0.984,0.153,0.946,0.000
blend,cap5_T1,0.362,0.000,0.648,0.771,0.951,0.352,0.827,0.000
blend,uncapped_T1,0.362,0.000,0.648,0.771,0.951,0.352,0.827,0.000
blend,cap5_T2,0.360,0.000,0.648,0.771,0.951,0.352,0.827,0.000
blend,uncapped_T2,0.366,0.000,0.648,0.771,0.951,0.352,0.827,0.000
blend,cap5_T1_classcenter,0.541,0.422,0.648,0.771,0.951,0.364,0.843,0.000


badnets 方法实际使用： {'hc': 'hc', 'birch': 'birch', 'meanshift': 'fallback_kmeans', 'dbscan': 'fallback_kmeans'}
blend 方法实际使用： {'hc': 'hc', 'birch': 'birch', 'meanshift': 'fallback_kmeans', 'dbscan': 'fallback_kmeans'}
主配置：损坏后图像嵌入 + 峰值 NTE 定向；第二列是 NTE 向量特征空间，仅供对照。
cap5_T1 是论文实现（官方 Notebook 的上限 5），uncapped_T1 是 Eq.9-11 的字面形式。
同一簇划分下，去除步骤使用 Eq.12 的恢复标签，所以 ASR/BA 会随归一化变化。


**关于 NTE 上限的代数事实。** 按 Eq. 9/10，T=1 时单个损坏的 NTE 是 `exp(s' − s) − 1`，
而 `s, s' ∈ [0,1]`，所以每个元素的取值不超过 `e − 1 ≈ 1.72`。因此 `cap=5`（乃至任何 ≥2 的
上限）在 T=1 下**不可能截断**：`cap5_T1` 与 `uncapped_T1` 两行逐位相同，是公式的直接推论，
不是实现 bug。官方 Notebook 用的是另一套归一化（`min(exp(2s), gap)`），上限在那里才有作用；
T=2 行把上界抬到 `e² − 1 ≈ 6.39`，才是检验“上限是否改变结论”的有效位置。


## 4. 数值摘要

In [6]:
print("自动摘要（只报告数值，不替读者下结论）")
for attack in ("badnets", "blend"):
    p = paper.get(attack, {})
    m = models.get(attack, {})
    rec = clustering.get(attack, {})
    c = rec.get("birch_embed", {})
    r = removal.get(attack, {})
    variants = rec.get("nte_variants") or {}
    best = max(variants.items(), key=lambda kv: kv[1].get("pdr", -1), default=("n/a", {}))
    print(f"\n[{attack}]")
    print(f"  BA    full={fmt(m.get('ba'))} vs 论文 No Defense={fmt(p.get('no_defense_ba'))}")
    print(f"  ASR   full={fmt(m.get('asr'))} vs 论文 No Defense={fmt(p.get('no_defense_asr'))}")
    print(f"  PDR   full={fmt(c.get('pdr'))} vs 论文 EBBA+={fmt(p.get('pdr'))}")
    print(f"  F1    full={fmt(c.get('f1'))} vs 论文 EBBA+={fmt(p.get('f1'))}")
    print(f"  PDR 细化（P0 诊断规则）={fmt(rec.get('refined', {}).get('pdr'))}；"
          f"FP={fmt(rec.get('refined', {}).get('fp'))}")
    print(f"  去除后池内 ASR={fmt(r.get('asr_after_pool'))}"
          f"（细化 {fmt(r.get('asr_after_pool_refined'))}）；"
          f"论文全测试集 ASR={fmt(p.get('ebba_plus_asr'))}")
    print(f"  聚类方法 PDR 极差={fmt(rec.get('pdr_spread'))}（论文 0.010）")
    if best[0] != "n/a":
        print(f"  最好的 NTE 变体={best[0]}：恢复率={fmt(best[1].get('recovery_poisoned'))}，"
              f"PDR={fmt(best[1].get('pdr'))}")


自动摘要（只报告数值，不替读者下结论）

[badnets]
  BA    full=0.924 vs 论文 No Defense=0.912
  ASR   full=0.986 vs 论文 No Defense=0.994
  PDR   full=0.891 vs 论文 EBBA+=0.960
  F1    full=0.935 vs 论文 EBBA+=0.960
  PDR 细化（P0 诊断规则）=0.256；FP=3.000
  去除后池内 ASR=0.109（细化 0.744）；论文全测试集 ASR=0.029
  聚类方法 PDR 极差=0.031（论文 0.010）
  最好的 NTE 变体=cap5_T1：恢复率=0.780，PDR=0.891

[blend]
  BA    full=0.942 vs 论文 No Defense=0.913
  ASR   full=0.999 vs 论文 No Defense=1.000
  PDR   full=0.648 vs 论文 EBBA+=0.950
  F1    full=0.771 vs 论文 EBBA+=0.940
  PDR 细化（P0 诊断规则）=0.171；FP=26.000
  去除后池内 ASR=0.352（细化 0.829）；论文全测试集 ASR=0.058
  聚类方法 PDR 极差=0.018（论文 0.010）
  最好的 NTE 变体=cap5_T1：恢复率=0.362，PDR=0.648


## 必须写清楚的差异

1. **OOD 图像不是论文的 Internet 图像。** full 档使用 CIFAR-100 test 10k + GTSRB test 12,630
   （resize 到 32×32，标签偏移 +100 以免与 CIFAR-100 冲突）。两者都与 CIFAR-10 类别不重叠，
   但仍不是论文所说的“任意互联网图片”。
2. **攻击实现接近论文，但不是官方代码。** BadNets 为右下 3×3 **黑块**（论文实现细节），
   Blend 使用 **hello kitty**（α=0.2；32×32 数组由 `scripts/prepare_trigger.py` 从本地图片生成）。
   投毒率 10%、dirty-label、目标类 0。若 `artifacts-full/data/blend_hello_kitty.npy` 不存在，
   代码退回程序化图案，并把这一选择写进训练配置哈希。
3. **候选池为 5,000 张中毒 + 1,000 张干净目标类（共 6,000）。** 论文的 PDR/F1 定义在全部
   中毒样本上；这里从 9,000 张非目标类测试图中抽 5,000 张，属于抽样估计，不是全测试集。
4. **聚类特征与定向规则是论文未写明的实现选择。** 主配置使用损坏后图像的 ResNet-18
   倒数第二层特征（80 种损坏取平均）做标准化 + Birch，并用簇内平均峰值 NTE 认定中毒簇；
   t-SNE 仅用于论文式可视化——在 6k 候选点上，它的精确解会随 GPU 批次差异改变 Birch 划分。
   NTE 向量 + 目标 logit 领先量仅作对照；`oracle` 只是上界。
5. **池内二次阈值与 NTE 扫描是诊断，不是论文方法。** 前者在已定向的中毒簇内对 peak NTE
   拟合 1/2 组分高斯混合（BIC 门控）以压低误报；后者扫描 Eq. 9/10 的上限、温度与类别中心化。
   若只有加上这些可调规则后数字才接近论文，应写成“论文方法 + 事后规则”，不能算作复现成绩。
6. **训练细节只覆盖公开配方。** ResNet-18 CIFAR 变体、SGD lr=0.01、momentum 0.9、
   weight decay 5e-4、batch 64、milestones 50/100/150、200 epochs、random crop + flip。
   论文未公开的初始化/数据顺序等差异仍存在。
7. **full 档没有覆盖 WaNet / FIBA / DUBA、GTSRB 主实验、ImageNet、语音/文本。**
   这些仍是 Tier C 的不可复现项。

因此，正确的结论句式是：

> 在 ResNet-18、完整 CIFAR-10、200 epochs、黑块/hello-kitty 触发器的独立设定下，
> BadNets/Blend 的 PDR/F1 达到 X；与论文的 0.96/0.94 相比差 Y。方向和量级（支持/不支持）如下……

而不是“逐位复现了论文的 96%”。
